# Diabetic Retinopathy Screening Agent
Vicheda Narith, Maanvi Sarwadi

## Setup & Dependencies

Run the following script to load packages and dependencies

`run script.sh`

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import requests
import os

In [2]:
UNCERTAINTY_THRESHOLD = 0.5
DR_GRADES = {
    0: "No apparent DR",
    1: "Mild NPDR",
    2: "Moderate NPDR",
    3: "Severe NPDR",
    4: "Proliferative DR"
}

## Load Pre-trained Models

In [3]:
# load EyePACS straight from huggingface
from datasets import load_dataset

eyepacs = load_dataset("ctmedtech/EYEPACS", split="train")
print(eyepacs)

README.md:   0%|          | 0.00/4.40k [00:00<?, ?B/s]

rm_images/Merged_Fundus_Images_with_Capt(…):   0%|          | 0.00/407k [00:00<?, ?B/s]

rm_images/left_preprocess_sample.jpg:   0%|          | 0.00/240k [00:00<?, ?B/s]

rm_images/right_preprocess_sample.jpg:   0%|          | 0.00/223k [00:00<?, ?B/s]

EYEPACS.zip:   0%|          | 0.00/6.48G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['image'],
    num_rows: 35111
})


In [4]:
# transforms to apply images
def get_transforms():
    return transforms.Compose(
        [
            transforms.Resize((512, 512)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

# dataset
class DRDataset(Dataset):
    def __init__(self, data_source, img_dir=None, transform=None):
        self.data_source = data_source
        self.img_dir = img_dir
        self.transform = transform or get_transforms()

    def __len__(self):
        return len(self.data_source)
    
    def __getitem__(self, idx):
        row = self.data_source[idx]
        if isinstance(row, pd.Series):
            row = row.to_dict()

        if isinstance(row, dict) and 'image' in row:
            img = row['image']
        else:
            img_name = row['image_name']
            img_path = os.path.join(self.img_dir, img_name)
            img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        label = row['label']
        return img, label

# create the dataset and load
dataset = DRDataset(eyepacs)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
print(eyepacs.features)
print(eyepacs[0].keys())

{'image': Image(mode=None, decode=True)}
dict_keys(['image'])


In [ ]:
from datasets import load_dataset
from torch.utils.data import random_split

# Load APTOS with labels — already 224x224, no preprocessing needed
aptos = load_dataset("bumbledeep/aptos", split="train")

print(aptos)
print(aptos.features)
print(aptos[0].keys())  # should show: image, label_code, label

README.md:   0%|          | 0.00/5.36k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/250M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3662 [00:00<?, ? examples/s]

Dataset({
    features: ['image', 'label_code', 'label'],
    num_rows: 3662
})
{'image': Image(mode=None, decode=True), 'label_code': Value('int64'), 'label': Value('string')}
dict_keys(['image', 'label_code', 'label'])


In [14]:
class DRDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data      = data
        self.transform = transform or get_transforms()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row   = self.data[idx]
        img   = row['image'].convert('RGB')
        label = row['label_code']  # integer 0–4
        if self.transform:
            img = self.transform(img)
        return img, label


# Train/val split (85/15)
n       = len(aptos)
n_train = int(0.85 * n)
n_val   = n - n_train

train_data, val_data = random_split(aptos, [n_train, n_val],
                                     generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(DRDataset(train_data), batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(DRDataset(val_data),   batch_size=32, shuffle=False, num_workers=0)

print(f'Train: {n_train} | Val: {n_val}')

Train: 3112 | Val: 550


In [ ]:
import timm

class DRVisionModel(nn.Module):
    def __init__(self, num_classes=5, dropout_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b3',
            pretrained=True,
            num_classes=0  # remove default head
        )
        feature_dim = self.backbone.num_features  # 1536

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

    def enable_mc_dropout(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = DRVisionModel().to(DEVICE)
print(f'Device: {DEVICE}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Device: cpu
Parameters: 11,485,741


In [ ]:
from tqdm import tqdm

def train(model, train_loader, val_loader, epochs=10):
    # Class weights to handle imbalance
    class_weights = torch.tensor([0.5, 1.5, 2.0, 2.5, 3.0]).to(DEVICE)
    criterion  = nn.CrossEntropyLoss(weight=class_weights)
    optimizer  = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')

    for epoch in range(epochs):
        # Train
        model.train()
        train_loss = 0
        for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1} [train]'):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for imgs, labels in tqdm(val_loader, desc=f'Epoch {epoch+1} [val]'):
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                val_loss += criterion(model(imgs), labels).item()

        train_loss /= len(train_loader)
        val_loss   /= len(val_loader)
        scheduler.step()

        print(f'Epoch {epoch+1}: train={train_loss:.4f}  val={val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'checkpoints/dr_best.pth')
            print('  → Checkpoint saved')

    print('Training complete.')

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)

train(model, train_loader, val_loader, epochs=10)

Epoch 1 [train]:   0%|          | 0/98 [00:00<?, ?it/s]